# APIs and GeoCoding and Plotting

## About

A simple set of cells to demonstrate how to use the Nominatim API to geocode an address and plot the result on a map.

In [7]:
import requests
import folium
import json

## Configure your data and API call

In [8]:
# ==================== CONFIG ====================
address = "Slater, Iowa"          # ← CHANGE THIS to any address/city you want!
user_agent = "MyGeospatialAPIDemo/1.0 (drfils@gmail.com)"  # ← REQUIRED by Nominatim policy — use your own name/email

# ==================== API CALL SETUP ====================
url = "https://nominatim.openstreetmap.org/search"

params = {
    "format": "json",          # ask for JSON
    "q": address,              # the search query
    "limit": 1,                # just the best match
    "addressdetails": 1        # extra info (optional but nice)
}

headers = {"User-Agent": user_agent}   # Nominatim requires this?


In [9]:
# ==================== API CALL ====================
response = requests.get(url, params=params, headers=headers)

# Basic error handling
if response.status_code != 200:
    raise Exception(f"API error: {response.status_code} — {response.text}")

data = response.json()

if not data:
    raise Exception("No results found for that address!")

result = data[0]  # first (and only) result

In [10]:

# ==================== PARSE RESULTS ====================
display_name = result["display_name"]
lat = float(result["lat"])
lon = float(result["lon"])

# Bounding box comes as [south, north, west, east] — all strings
bbox = result["boundingbox"]
south, north, west, east = map(float, bbox)

print(f"✅ Found: {display_name}")
print(f"   Coordinates: {lat:.4f}, {lon:.4f}")
print(f"   Bounding box: South={south:.4f}, North={north:.4f}, West={west:.4f}, East={east:.4f}")

# ==================== CONVERT BBOX → WKT ====================
# WKT Polygon (counter-clockwise order, closed ring)
wkt_polygon = (
    f"POLYGON(({west} {south}, "
    f"{east} {south}, "
    f"{east} {north}, "
    f"{west} {north}, "
    f"{west} {south}))"
)

print("\n📍 WKT Polygon (ready for PostGIS, SQL, or any GIS tool):")
print(wkt_polygon)

✅ Found: Slater, Palestine Township, Story County, Iowa, United States
   Coordinates: 41.8824, -93.6837
   Bounding box: South=41.8681, North=41.8889, West=-93.6981, East=-93.6736

📍 WKT Polygon (ready for PostGIS, SQL, or any GIS tool):
POLYGON((-93.6980908 41.8680843, -93.6735998 41.8680843, -93.6735998 41.8888519, -93.6980908 41.8888519, -93.6980908 41.8680843))


In [11]:
# ==================== DISPLAY INTERACTIVE MAP (Leaflet) ====================
# Center the map on the result
m = folium.Map(location=[lat, lon], zoom_start=12, tiles="OpenStreetMap")

# 1. Marker at the exact point
folium.Marker(
    location=[lat, lon],
    popup=f"<b>{display_name}</b><br>Lat: {lat:.4f}<br>Lon: {lon:.4f}",
    tooltip="Click for details",
    icon=folium.Icon(color="red", icon="info-sign")
).add_to(m)

# 2. Bounding box as a semi-transparent rectangle (super clean for Leaflet)
folium.Rectangle(
    bounds=[[south, west], [north, east]],
    color="#3388ff",
    weight=3,
    fill=True,
    fill_color="#3388ff",
    fill_opacity=0.2,
    popup="Bounding Box from Nominatim"
).add_to(m)

# Optional: add the WKT as a tooltip on the map
folium.Marker(
    location=[north, east],
    icon=folium.DivIcon(html=f"<div style='font-size:10px; color:blue;'>WKT ready!</div>")
).add_to(m)

# Show the map right in the notebook!
m

## This version uses GeoJSON

In [12]:


# ==================== PARSE ====================
display_name = result["display_name"]
lat = float(result["lat"])
lon = float(result["lon"])
bbox = result["boundingbox"]           # [south, north, west, east] as strings
south, north, west, east = map(float, bbox)

print(f"✅ {display_name}")
print(f"   Point: {lat:.5f}, {lon:.5f}")
print(f"   BBox:  S={south:.5f}  N={north:.5f}  W={west:.5f}  E={east:.5f}\n")

# ==================== BUILD GEOJSON ====================
geojson_feature = {
    "type": "Feature",
    "properties": {
        "name": display_name,
        "place_type": result.get("type", "unknown"),
        "osm_id": result.get("osm_id"),
        "category": result.get("category"),
        "importance": result.get("importance")
    },
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [west, south],   # bottom-left
            [east, south],   # bottom-right
            [east, north],   # top-right
            [west, north],   # top-left
            [west, south]    # close the ring
        ]]
    }
}

# Pretty-print the GeoJSON
print("📦 GeoJSON Feature (Polygon from bounding box):")
print(json.dumps(geojson_feature, indent=2))

# ==================== INTERACTIVE MAP ====================
m = folium.Map(location=[lat, lon], zoom_start=13, tiles="OpenStreetMap")

# Center marker
folium.Marker(
    [lat, lon],
    popup=f"<b>{display_name}</b><br>Lat/Lon: {lat:.5f}, {lon:.5f}",
    tooltip="Nominatim center point",
    icon=folium.Icon(color="red", icon="star")
).add_to(m)

# Add the bounding box as GeoJSON layer — styled nicely
folium.GeoJson(
    geojson_feature,
    name="Nominatim Bounding Box",
    style_function=lambda x: {
        "fillColor": "#3388ff",
        "color": "#3388ff",
        "weight": 3,
        "fillOpacity": 0.18,
    },
    tooltip=folium.GeoJsonTooltip(fields=["name", "place_type", "category"])
).add_to(m)

folium.LayerControl().add_to(m)

# Show it!
m

✅ Slater, Palestine Township, Story County, Iowa, United States
   Point: 41.88241, -93.68368
   BBox:  S=41.86808  N=41.88885  W=-93.69809  E=-93.67360

📦 GeoJSON Feature (Polygon from bounding box):
{
  "type": "Feature",
  "properties": {
    "name": "Slater, Palestine Township, Story County, Iowa, United States",
    "place_type": "administrative",
    "osm_id": 128329,
    "category": null,
    "importance": 0.46256279691251534
  },
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [
        [
          -93.6980908,
          41.8680843
        ],
        [
          -93.6735998,
          41.8680843
        ],
        [
          -93.6735998,
          41.8888519
        ],
        [
          -93.6980908,
          41.8888519
        ],
        [
          -93.6980908,
          41.8680843
        ]
      ]
    ]
  }
}
